# 003 · Properties

> **Syntax › Classes & OOP** · Python 3.14

C# developers wrap almost every field in a property from day one. Python takes the opposite approach: start with **plain attributes**, and only turn one into a **property** when you need logic — callers never notice the change.

**In this notebook you will learn:**

- `@property` for getters, plus `@x.setter` and `@x.deleter`
- read-only and computed properties (C#'s `{ get; }` and `=>` expression-bodied members)
- validating values in a setter
- why Python has no auto-properties — and doesn't need them
- `functools.cached_property` as a lightweight `Lazy<T>`

## C# mental model

| C# | Python |
|---|---|
| `public string Name { get; set; }` (auto-property) | plain attribute `self.name = name` |
| `public double Area => W * H;` | `@property def area(self): return ...` |
| `{ get; private set; }` | `@property` + "private" backing attribute `_x` |
| `set { if (value < 0) throw ...; _x = value; }` | `@x.setter` with validation |
| `Lazy<T>` / lazily computed property | `@functools.cached_property` |

The key mental shift: in Python, `obj.x` syntax looks identical whether `x` is a plain attribute or a property. So switching from one to the other is **not a breaking change** — unlike C#, where changing a field to a property breaks binary compatibility.

## Start with plain attributes

In C# you write auto-properties so you can add logic later without breaking callers. In Python you simply use a public attribute:

```csharp
// C#
public class Product
{
    public string Name { get; set; }
    public decimal Price { get; set; }
}
```

In [ ]:
class Product:
    def __init__(self, name, price):
        self.name = name        # no getters/setters needed
        self.price = price

p = Product("pen", 1.5)
p.price = 2.0
print(p.name, p.price)

Writing Java/C#-style `get_price()` / `set_price()` methods is considered un-Pythonic. If you later need validation, convert `price` into a property — the code above keeps working unchanged.

## `@property` — a computed getter

Decorating a method with `@property` makes it readable as an attribute: no parentheses at the call site. A property with only a getter is **read-only**.

```csharp
// C#
public class Rectangle(double width, double height)
{
    public double Width { get; } = width;
    public double Height { get; } = height;
    public double Area => Width * Height;          // expression-bodied, read-only
}
```

In [ ]:
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    @property
    def area(self):
        return self.width * self.height

r = Rectangle(2, 3)
print(r.area)          # 6 — accessed like an attribute, no ()
r.width = 10
print(r.area)          # 30 — recomputed on every access

Assigning to a getter-only property fails at runtime with `AttributeError` (C# would give a compile error):

In [ ]:
try:
    r.area = 100
except AttributeError as e:
    print(f"{type(e).__name__}: {e}")

## Getter + setter with validation

To add a setter, decorate a second method **with the same name** using `@<name>.setter`. The value is stored in a "private" backing attribute, conventionally prefixed with `_` *(see Classes & OOP › 004 · Encapsulation and Naming Conventions)*.

```csharp
// C#
private decimal _price;
public decimal Price
{
    get => _price;
    set
    {
        ArgumentOutOfRangeException.ThrowIfNegative(value);
        _price = value;
    }
}
```

In [ ]:
class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price            # goes through the setter — validated too!

    @property
    def price(self):
        return self._price

    @price.setter
    def price(self, value):
        if value < 0:
            raise ValueError(f"price must be >= 0, got {value}")
        self._price = value

p = Product("pen", 1.5)
p.price = 2.0                         # same syntax as before — callers didn't change
print(p.price)

try:
    p.price = -5
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

Note that `__init__` assigns `self.price` (not `self._price`) so the constructor gets the same validation as later updates.

⚠️ **Gotcha:** inside the property, use the **backing** name. Writing `self.price = value` inside the `price` setter calls the setter again, recursing until `RecursionError`.

In [ ]:
class Broken:
    @property
    def value(self):
        return self.value             # calls the getter again... forever

try:
    Broken().value
except RecursionError as e:
    print(f"{type(e).__name__}: {e}")

⚠️ **Gotcha:** the setter and getter must have the **same function name** as the property. `@price.setter` returns a *new* property object that includes both functions, and it must replace the name `price` in the class body.

## Read-only from the outside: `{ get; private set; }`

Python has no access modifiers, so "private set" is expressed by exposing a getter-only property and updating the underscore-prefixed backing attribute internally.

```csharp
// C#
public int Balance { get; private set; }
public void Deposit(int amount) => Balance += amount;
```

In [ ]:
class Account:
    def __init__(self):
        self._balance = 0

    @property
    def balance(self):
        return self._balance

    def deposit(self, amount):
        self._balance += amount        # internal code writes the backing attribute

acc = Account()
acc.deposit(50)
print(acc.balance)

try:
    acc.balance = 1_000_000
except AttributeError as e:
    print(f"{type(e).__name__}: {e}")

The `_balance` attribute is still technically writable (`acc._balance = 1`). The underscore is a strong convention meaning "internal — hands off", not enforced protection.

## Deleter

A property can also define what `del obj.attr` does. This is rare — there is no C# counterpart — but it occasionally helps for resetting cached state.

In [ ]:
class Profile:
    def __init__(self, nickname):
        self._nickname = nickname

    @property
    def nickname(self):
        return self._nickname if self._nickname is not None else "<anonymous>"

    @nickname.setter
    def nickname(self, value):
        self._nickname = value

    @nickname.deleter
    def nickname(self):
        print("resetting nickname")
        self._nickname = None

pr = Profile("bob")
del pr.nickname
print(pr.nickname)

## Computed properties that depend on each other

Properties are a natural fit for derived values, such as a temperature stored once and exposed in two units. Setting either property keeps both consistent:

In [ ]:
class Temperature:
    def __init__(self, celsius=0.0):
        self.celsius = celsius

    @property
    def fahrenheit(self):
        return self.celsius * 9 / 5 + 32

    @fahrenheit.setter
    def fahrenheit(self, value):
        self.celsius = (value - 32) * 5 / 9

t = Temperature(100)
print(t.fahrenheit)     # 212.0
t.fahrenheit = 32
print(t.celsius)        # 0.0

**Guideline:** a property should feel like reading a field — cheap and side-effect free. If the work is expensive or might fail (database calls, I/O), use a regular method so callers can see that work is happening (same advice as in .NET's design guidelines).

## Lazy, cached properties — `functools.cached_property`

For an expensive value that never changes once computed, `@cached_property` computes it on first access and stores the result on the instance. It plays the role of `Lazy<T>` (or a manually cached `??=` property) in C#.

```csharp
// C#
private readonly Lazy<int> _wordCount;
public Document(string text) => _wordCount = new(() => text.Split(' ').Length);
public int WordCount => _wordCount.Value;
```

In [ ]:
from functools import cached_property

class Document:
    def __init__(self, text):
        self.text = text

    @cached_property
    def word_count(self):
        print("  (computing...)")
        return len(self.text.split())

doc = Document("the quick brown fox")
print(doc.word_count)    # computes
print(doc.word_count)    # cached — no "computing" this time

Unlike `Lazy<T>`, the cached value can be invalidated by deleting it, after which it is recomputed on next access. It is also stored in the instance's `__dict__`, so it does **not** work with classes that use `__slots__`. `cached_property` is not thread-safe in the `Lazy<T>` sense (two threads might compute it twice).

In [ ]:
doc.text = "hello world"
print(doc.word_count)    # still the old value: 4 — the cache doesn't know text changed
del doc.word_count       # invalidate
print(doc.word_count)    # recomputed: 2

## Properties are class-level objects

Under the hood, `property` is a built-in class that implements the *descriptor protocol*; the decorators simply build a `property(fget, fset, fdel)` object stored on the class. You may see the older, non-decorator spelling in existing code:

In [ ]:
class Circle:
    def __init__(self, radius):
        self.radius = radius

    def _get_diameter(self):
        return self.radius * 2

    diameter = property(_get_diameter)    # equivalent to @property

c = Circle(3)
print(c.diameter)
print(type(Circle.__dict__["diameter"]))  # <class 'property'>

## Summary

| Task | C# | Python |
|---|---|---|
| Simple data member | `{ get; set; }` auto-property | plain attribute |
| Computed read-only | `public T X => ...;` | `@property` |
| Getter + setter | `get { } set { }` | `@property` + `@x.setter` |
| Validation | logic in `set` | logic in `@x.setter`, assign `self.x` in `__init__` |
| Public get, private set | `{ get; private set; }` | `@property` over `self._x` |
| Delete behaviour | (none) | `@x.deleter` |
| Lazy value | `Lazy<T>` | `@functools.cached_property` |
| Field → property later | breaking change | transparent to callers |

## Exercises

1. Write a `Person` class with a `name` attribute and an `age` property whose setter rejects negative numbers or ages above 150. Make sure the constructor validates too.
2. Add a read-only `is_adult` property (age >= 18).
3. Write a `Circle` class with a `radius` attribute and a `diameter` property **with a setter** that updates the radius.
4. Create a class with a `cached_property` that returns a sorted copy of a list attribute. Show how to refresh it after the list changes.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1, 2
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age                 # runs the setter

    @property
    def age(self):
        return self._age

    @age.setter
    def age(self, value):
        if not 0 <= value <= 150:
            raise ValueError(f"invalid age: {value}")
        self._age = value

    @property
    def is_adult(self):
        return self.age >= 18

# 3
class Circle:
    def __init__(self, radius):
        self.radius = radius

    @property
    def diameter(self):
        return self.radius * 2

    @diameter.setter
    def diameter(self, value):
        self.radius = value / 2

# 4
from functools import cached_property

class Scores:
    def __init__(self, values):
        self.values = values

    @cached_property
    def ranked(self):
        return sorted(self.values, reverse=True)

s = Scores([3, 9, 1])
print(s.ranked)
s.values.append(10)
del s.ranked                         # invalidate the cache
print(s.ranked)
```

</details>